# Fine-Tuning Whisper LoRA v3 — Real Stochastic Regularization

**What changed from v2, and why:**

Your v2 run (r=16 + SpecAugment + early stopping) landed at 23.693% held-out WER — statistically the same as your original v1 result (23.7%). Doubling LoRA rank (8→16) moved the number by ~0.007 points, which means **you are not capacity-bound** — increasing rank/epochs further is very unlikely to help.

Digging into the v2 notebook found the likely reason SpecAugment didn't actually prevent overfitting: it was applied **once**, inside `dataset.map()`, before training started — so the model saw the exact same "randomly" masked spectrogram every single epoch for 15 epochs. That's not real SpecAugment regularization, it's a fixed data perturbation. Same story for the speed/pitch augmentation: every row got the *exact same* fixed parameters (rate=1.1, n_steps=1.5), not randomized diversity.

**Two fixes in this notebook:**
1. **SpecAugment moved into the data collator**, so masking is randomly regenerated on every batch/epoch — genuine stochastic regularization.
2. **Speed/pitch augmentation parameters are now randomized per training row** (instead of one fixed value for everyone), for a bit more acoustic diversity without exploding your compute budget.

LoRA rank is kept at **r=8** by default (configurable below) since v2 already showed r=16 buys you nothing — this run isolates the regularization fix as the only real change, so if it helps, you'll know *why*.

Run cells in order, top to bottom. Same Drive folder structure as your existing notebooks.

## Step 1 - GPU check
Runtime -> Change runtime type -> T4 GPU -> Save.

In [ ]:
!pip install -q transformers datasets evaluate jiwer accelerate soundfile librosa peft torchao --upgrade
!pip install -q -U datasets pyarrow accelerate

## Step 2 - Connect to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
base_path = "/content/drive/MyDrive/capstone-cleft-speech-data"
wav_folder = f"{base_path}/converted_wav"
csv_path = f"{base_path}/tracking/dataset.csv"
print("WAV files found:", len(os.listdir(wav_folder)))

## Step 3 - Load your dataset and split it

**Same `random_state=42`, same `test_size=0.15` as v1/v2** — this recreates the identical held-out set. Do not change this line, or you'll silently break comparability with your previous results (and risk leakage if held-out files end up in a different split).

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(csv_path)
df['wav_filename'] = df['filename'].apply(lambda x: os.path.splitext(x)[0] + ".wav")
df['wav_path'] = df['wav_filename'].apply(lambda x: os.path.join(wav_folder, x))
df = df[df['wav_path'].apply(os.path.exists)].reset_index(drop=True)
print("Usable recordings:", len(df))

train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
print(f"Training on: {len(train_df)} recordings (before augmentation)")
print(f"Validating on: {len(val_df)} recordings (held out - never augmented, never seen in training)")

## Step 4 - Data augmentation (training set only, randomized parameters)

Still 2 extra copies per training recording (same compute budget as v2), but now each copy gets its **own random draw** for speed/pitch instead of everyone getting an identical fixed transform. This gives more genuine acoustic diversity per file for roughly the same training set size and runtime as before.

Validation set is deliberately left untouched, same as before.

In [ ]:
import librosa
import soundfile as sf
import numpy as np
import os
import pandas as pd

np.random.seed(42)  # reproducible augmentation choices

augmented_folder = f"{base_path}/augmented_wav_v3"
os.makedirs(augmented_folder, exist_ok=True)
augmented_rows = []

for idx, row in train_df.iterrows():
    y, sr = librosa.load(row['wav_path'], sr=16000)

    # Version 1: randomized speed perturbation (was fixed at 1.1x in v2)
    speed_rate = np.random.uniform(0.9, 1.15)
    y_speed = librosa.effects.time_stretch(y, rate=speed_rate)
    speed_name = row['wav_filename'].replace(".wav", "_aug_speed.wav")
    speed_path = os.path.join(augmented_folder, speed_name)
    sf.write(speed_path, y_speed, sr)
    augmented_rows.append({"wav_path": speed_path, "intended_text": row['intended_text']})

    # Version 2: randomized pitch shift (was fixed at +1.5 semitones in v2)
    pitch_steps = np.random.uniform(-2.0, 2.0)
    y_pitch = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch_steps)
    pitch_name = row['wav_filename'].replace(".wav", "_aug_pitch.wav")
    pitch_path = os.path.join(augmented_folder, pitch_name)
    sf.write(pitch_path, y_pitch, sr)
    augmented_rows.append({"wav_path": pitch_path, "intended_text": row['intended_text']})

augmented_df = pd.DataFrame(augmented_rows)

train_df_full = pd.concat([
    train_df[['wav_path', 'intended_text']],
    augmented_df], ignore_index=True)

print(f"Original training recordings: {len(train_df)}")
print(f"Augmented recordings added: {len(augmented_df)}")
print(f"Total training set size now: {len(train_df_full)}")

## Step 5 - Load the Whisper model and processor

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name, language="English", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.language = "english"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

## Step 6 - Apply LoRA (r kept at 8 — capacity is not the bottleneck)

Your v2 A/B test (r=8 vs r=16) already showed rank isn't the lever that matters. Kept configurable here in case you want to re-check later, but r=8 is the recommended default for this run so the regularization fix is the only variable that changed.

In [ ]:
from peft import LoraConfig, get_peft_model

LORA_R = 8  # v2 test showed r=16 gave no meaningful improvement over r=8 - keep at 8

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 7 - Prepare the data for training

**Key change:** `prepare_train_example` no longer bakes in a fixed SpecAugment mask. It just extracts raw features. The masking now happens fresh every batch, inside the data collator (Step 8) — that's what makes it real per-epoch regularization instead of a one-time static transform.

In [ ]:
from datasets import Dataset, Audio

def to_hf_dataset(dataframe):
    ds = Dataset.from_dict({
        "audio": list(dataframe['wav_path']),
        "text": list(dataframe['intended_text'])
    })
    ds = ds.cast_column("audio", Audio(sampling_rate=16000))
    return ds

def prepare_train_example(batch):
    audio = batch["audio"]
    features = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["input_features"] = features  # raw features - SpecAugment applied later, per-batch, in the collator
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

def prepare_eval_example(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

train_dataset = to_hf_dataset(train_df_full)
train_dataset = train_dataset.map(prepare_train_example, remove_columns=train_dataset.column_names)

val_dataset_hf = to_hf_dataset(val_df.rename(columns={"intended_text": "text"}) if "text" not in val_df.columns else val_df)
val_dataset_hf = to_hf_dataset(val_df.assign(text=val_df['intended_text']))
val_dataset_hf = val_dataset_hf.map(prepare_eval_example, remove_columns=val_dataset_hf.column_names)

print("Train examples:", len(train_dataset))
print("Val examples:", len(val_dataset_hf))

## Step 8 - Data collator with **dynamic, per-batch SpecAugment**

This is the actual fix. `spec_augment()` is the same masking function as v2 — but instead of running once inside `.map()`, it now runs **inside `__call__`, every time the Trainer pulls a batch**, and only on the training collator (`apply_spec_augment=True`). The eval collator has it turned off, so validation always sees clean, unaugmented features — exactly as before.

In [ ]:
import torch
import numpy as np
from dataclasses import dataclass
from typing import Any, Dict, List, Union

def spec_augment(features, freq_mask_width=8, time_mask_width=25, n_freq_masks=2, n_time_masks=2):
    features = features.copy()
    n_mels, n_frames = features.shape
    for _ in range(n_freq_masks):
        f = np.random.randint(0, freq_mask_width)
        f0 = np.random.randint(0, max(1, n_mels - f))
        features[f0:f0+f, :] = 0
    for _ in range(n_time_masks):
        t = np.random.randint(0, time_mask_width)
        t0 = np.random.randint(0, max(1, n_frames - t))
        features[:, t0:t0+t] = 0
    return features

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    apply_spec_augment: bool = False

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        if self.apply_spec_augment:
            for f in features:
                arr = np.array(f["input_features"])
                f["input_features"] = spec_augment(arr)  # fresh random mask, every call

        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

train_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor, apply_spec_augment=True)
eval_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor, apply_spec_augment=False)

## Step 9 - WER metric for tracking progress during training

In [ ]:
import evaluate
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

## Step 10 - Training settings

Same safe checkpoint/early-stopping setup as your fixed v2 run (`load_best_model_at_end=True`, `metric_for_best_model="wer"`, `EarlyStoppingCallback`) — this part of v2 was correct and stays as-is. One important note: the `Seq2SeqTrainer` only takes a single `data_collator`, so we pass the training collator here and handle eval separately with a plain `Trainer.evaluate`-compatible workaround by swapping the collator right before `.evaluate()` calls Hugging Face triggers internally — in practice this works fine because the Trainer uses `args.eval_strategy` calls under the hood with the same collator object. To guarantee eval never gets augmented, we use a small wrapper collator that checks `model.training`.

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

@dataclass
class ModeAwareCollator:
    train_collator: Any
    eval_collator: Any
    trainer_ref: Any = None

    def __call__(self, features):
        is_training = self.trainer_ref is not None and self.trainer_ref.model.training
        collator = self.train_collator if is_training else self.eval_collator
        return collator(features)

mode_aware_collator = ModeAwareCollator(train_collator=train_collator, eval_collator=eval_collator)

output_dir = f"{base_path}/finetuned_whisper_lora_v3"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=30,
    num_train_epochs=15,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    generation_max_length=128,
    logging_steps=10,
    save_total_limit=3,
    report_to=[],
    remove_unused_columns=False,
    label_names=["labels"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset_hf,
    data_collator=mode_aware_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
mode_aware_collator.trainer_ref = trainer

## Step 11 - Train

Same rough runtime ballpark as v2 (30-60 min on a T4). Watch the per-epoch WER column in the log output — with real per-epoch SpecAugment, you should see the overfitting gap (train loss dropping while val WER worsens) show up later or less severely than v2's epoch 7-8 turnaround. Do not close the tab or let the session idle.

In [ ]:
trainer.train()

## Step 12 - Verify training actually changed the model before saving

**Reminder:** this `conv1.weight` check is a weak sanity check only — it will look "unchanged" for reasons unrelated to LoRA success. Don't rely on it alone; Step 13's behavioral test is the one that actually matters.

In [ ]:
base_check = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
trained_weight = model.base_model.model.model.encoder.conv1.weight.detach().cpu()
fresh_weight = base_check.model.encoder.conv1.weight.detach().cpu()
identical = torch.equal(trained_weight, fresh_weight)
print("conv1.weight identical to pretrained base (expected True for LoRA - this check does not indicate failure):", identical)

## Step 13 - Behavioral verification test (the one that actually matters)

In [ ]:
import librosa

test_file = f"{base_path}/converted_wav/carrot.wav"

audio, sr = librosa.load(test_file, sr=16000)
inputs = processor.feature_extractor(audio, sampling_rate=16000, return_tensors="pt").input_features.to(model.device)

model.eval()
with torch.no_grad():
    with_adapter_ids = model.generate(inputs, max_new_tokens=50)
    with_adapter_text = processor.tokenizer.decode(with_adapter_ids[0], skip_special_tokens=True)

    with model.disable_adapter():
        without_adapter_ids = model.generate(inputs, max_new_tokens=50)
        without_adapter_text = processor.tokenizer.decode(without_adapter_ids[0], skip_special_tokens=True)

print("WITH LoRA adapter:", with_adapter_text)
print("WITHOUT LoRA adapter (base model behavior):", without_adapter_text)
print("Are they different?", with_adapter_text != without_adapter_text)

## Step 14 - Save your LoRA fine-tuned model (v3)

In [ ]:
model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)
print(f"LoRA fine-tuned model (v3) saved to: {output_dir}")

## Step 15 - Next: run the Error Analysis notebook

Load this v3 model in your Baseline WER notebook the same way as v2 (`PeftModel.from_pretrained` + `merge_and_unload()`, pointing at `finetuned_whisper_lora_v3` instead of `_v2`), evaluate on the same 54-file held-out set, save the results CSV, then run **`Error_Analysis_WER_Breakdown.ipynb`** against that CSV. That notebook will tell you:
- whether the new number is a real improvement or within noise (bootstrap confidence interval)
- micro vs. macro WER (whether your long sentences are dominating the pooled number)
- which phoneme categories / item types are still driving errors, so your next move (more targeted recordings) is aimed at the right place